In [1]:
!pip install streamlit pandas plotly pyngrok -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 6.2 MB/s eta 0:00:00


In [2]:
!ngrok authtoken "2uLQOD5sKv3LcAKpXd2ADIpHpBN_68DP5ovgwK23cW4VSHT4t"

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml



## 📁 필요한 데이터 파일 업로드

아래 셀을 실행하고 다음 3개 파일을 업로드하세요:

- `merged_with_sentiment_and_embedding.pkl` (임베딩 포함 메인 리뷰 데이터)
- `감성+정형_통합_최종_수정.csv` (정형 + 감성 통합 데이터)
- `car_keywords.csv` (차종별 핵심 키워드 데이터)


In [3]:
from google.colab import files

print("📁 아래 파일들을 업로드해주세요:")
print("- merged_with_sentiment_and_embedding.pkl")
print("- 감성+정형_통합_최종_수정.csv")
print("- car_keywords.csv")

uploaded = files.upload()

📁 아래 파일들을 업로드해주세요:
- merged_with_sentiment_and_embedding.pkl
- 감성+정형_통합_최종_수정.csv
- car_keywords.csv


Saving car_keywords.csv to car_keywords.csv
Saving merged_with_sentiment_and_embedding.pkl to merged_with_sentiment_and_embedding.pkl
Saving 감성+정형_통합_최종_수정.csv to 감성+정형_통합_최종_수정.csv


In [4]:
!pip install koreanize_matplotlib
!pip install matplotlib
!pip install konlpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 97.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.1/494.1 kB 36.2 MB/s eta 0:00:00


In [5]:
!pip install streamlit pandas plotly pyngrok faiss-cpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 13.5 MB/s eta 0:00:00


In [6]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import pickle
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import faiss
from wordcloud import WordCloud
from collections import Counter
import os
import koreanize_matplotlib
from konlpy.tag import Okt


# 한글 폰트 설정
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
os.system('apt-get update -qq')
os.system('apt-get install -y fonts-nanum')
os.system('fc-cache -fv')
os.system('fc-list :lang=ko')

# 폰트 경로 설정 (시스템에 따라 다를 수 있음)
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'  # 기본 경로
if not os.path.exists(font_path):
    # 윈도우 환경 경로 시도
    windows_font_path = 'C:/Windows/Fonts/malgun.ttf'
    if os.path.exists(windows_font_path):
        font_path = windows_font_path

plt.rc('font', family='NanumGothic')

# 페이지 설정
st.set_page_config(page_title="🚘 Car.You | 당신을 위한 차량 추천 서비스", layout="wide")

# 대시보드 타이틀
st.title("🚘 Car.You: 당신에게 맞춘 차량 데이터 분석 & 추천 시스템")
st.markdown("##### 후기를 읽지 않아도, 당신의 기준에 맞는 차량을 쉽게 찾을 수 있도록.")

@st.cache_data
def load_data():
    return pd.read_csv("감성+정형_통합_최종_수정.csv")

@st.cache_resource
def load_model():
    return SentenceTransformer("snunlp/KR-SBERT-V40K-klueNLI-augSTS")

@st.cache_data
def load_embeddings():
    try:
        with open("merged_with_sentiment_and_embedding.pkl", "rb") as f:
            return pickle.load(f)
    except:
        st.error("임베딩 파일을 불러올 수 없습니다.")
        return None

@st.cache_data
def load_keywords():
    try:
        keyword_df = pd.read_csv("car_keywords.csv", encoding="utf-8-sig")
        car_keywords = {}
        for car_model in keyword_df['차종'].unique():
            keywords = keyword_df[keyword_df['차종'] == car_model]['키워드'].tolist()
            car_keywords[car_model] = keywords
        return car_keywords
    except:
        st.warning("핵심 키워드 파일을 불러올 수 없습니다.")
        return {}

# 데이터 로드
df = load_data()
model = load_model()
embedded_df = load_embeddings()
car_keywords = load_keywords()
#tab1, tab2
tab3, tab4 = st.tabs([
    "📈 Car.You | 데이터 통합 인사이트",
    "🚘 Car.You 추천 | 나에게 딱 맞는 차는?"
])


# ----------------------- 통합 분석 -----------------------
with tab3:
    st.header("📈 Car.You | 데이터 통합 인사이트")

    # 감성점수를 차종별로 평균내고 상위 N개 추출
    st.subheader("1. 감성점수 기준 Top 10 차종")
    top_n = 10  # 원하는 Top N 개수 설정
    top_models = df.groupby("차종")["final_sentiment_score_new"].mean().sort_values(ascending=False).head(top_n).index

    # Top N 차종 데이터만 필터링
    top_df = df[df["차종"].isin(top_models)]

    # 1. 가격과 감성 점수
    st.subheader("① 감성점수 vs 가격")
    fig1 = px.scatter(
        top_df,
        x="가격 (만원)",
        y="final_sentiment_score_new",
        color="제조사",
        hover_name="차종",
        title="Top 감성차량의 가격 vs 감성점수"
    )
    st.plotly_chart(fig1, use_container_width=True, key="chart_14")


    # 2. 연비와 감성 점수
    st.subheader("② 감성점수 vs 연비")
    ice_df = top_df[top_df["연비(km/L)"].notna()]
    fig2 = px.scatter(
        ice_df,
        x="연비(km/L)",
        y="final_sentiment_score_new",
        color="제조사",
        hover_name="차종",
        title="Top 감성차량의 연비 vs 감성점수"
    )
    st.plotly_chart(fig2, use_container_width=True, key="chart_15")


    # 평균 기준으로 차종별 정리
    top_agg = top_df.groupby(["차종", "제조사"]).agg({
        "가격 (만원)": "mean",
        "최고출력 (마력)": "mean"
    }).reset_index()

    fig4 = px.bar(
        top_agg,
        x="차종",
        y="가격 (만원)",
        color="제조사",
        title="Top 감성차량 평균 가격 비교"
    )
    st.plotly_chart(fig4, use_container_width=True, key="chart_17")

    fig5 = px.bar(
        top_agg,
        x="차종",
        y="최고출력 (마력)",
        color="제조사",
        title="Top 감성차량 평균 출력 비교"
    )
    st.plotly_chart(fig5, use_container_width=True, key="chart_18")


# ----------------------- 차량 추천 시스템 -----------------------
with tab4:
    st.header("🚘 Car.You 추천 | 나에게 딱 맞는 차는?")

    # 사용자 입력값 파싱 함수
    def parse_factors(user_input):
        if ',' in user_input or ' ' in user_input:
            raw_factors = user_input.replace(',', ' ').split()
        else:
            raw_factors = list(user_input)
        return sorted(set([char for char in raw_factors if char in ['0', '1', '2']]))

    # 사이드바에 필터링 옵션
    st.sidebar.subheader("⚙️ 필터링 옵션")

    # 고려사항 선택
    st.sidebar.markdown("**🚗 차량 구매시 고려사항을 선택해주세요:**")
    price_factor = st.sidebar.checkbox("💰 가격", value=True)
    fuel_factor = st.sidebar.checkbox("⛽ 연료타입", value=True)
    body_factor = st.sidebar.checkbox("🚙 바디타입", value=True)

    # 선택한 고려사항 기반으로 필터링
    chosen_factors = []
    if price_factor:
        chosen_factors.append('0')
    if fuel_factor:
        chosen_factors.append('1')
    if body_factor:
        chosen_factors.append('2')

    # 데이터 복사
    if embedded_df is not None:
        filtered_df = embedded_df.copy()
    else:
        filtered_df = df.copy()

    # 1. 가격 필터링
    if '0' in chosen_factors:
        st.sidebar.markdown("### 💰 가격 범위 설정")
        min_price = st.sidebar.number_input(
            "최소 가격 (만원):",
            min_value=int(filtered_df["가격 (만원)"].min()),
            max_value=int(filtered_df["가격 (만원)"].max()),
            value=int(filtered_df["가격 (만원)"].min())
        )
        max_price = st.sidebar.number_input(
            "최대 가격 (만원):",
            min_value=int(filtered_df["가격 (만원)"].min()),
            max_value=int(filtered_df["가격 (만원)"].max()),
            value=int(filtered_df["가격 (만원)"].max())
        )
        filtered_df = filtered_df[
            (filtered_df["가격 (만원)"] >= min_price) &
            (filtered_df["가격 (만원)"] <= max_price)
        ]

    # 2. 연료타입 필터링
    if '1' in chosen_factors:
        st.sidebar.markdown("### ⛽ 연료타입 선택")
        fuel_map = {0: "가솔린/디젤", 1: "하이브리드", 2: "전기"}
        fuel_options = [0, 1, 2]
        fuel_choices = []

        for fuel_type in fuel_options:
            if st.sidebar.checkbox(fuel_map[fuel_type], value=True):
                fuel_choices.append(fuel_type)

        if fuel_choices:
            filtered_df = filtered_df[filtered_df["연료타입"].isin(fuel_choices)]

    # 3. 바디타입 필터링
    if '2' in chosen_factors:
        st.sidebar.markdown("### 🚙 바디타입 선택")
        body_map = {0: "승용", 1: "SUV", 2: "MPV"}
        body_options = [0, 1, 2]
        body_choices = []

        for body_type in body_options:
            if st.sidebar.checkbox(body_map[body_type], value=True):
                body_choices.append(body_type)

        if body_choices:
            filtered_df = filtered_df[filtered_df["바디타입"].isin(body_choices)]

    # 필터링 결과 표시
    st.subheader("✅ 필터링 결과")
    num_unique_models = len(filtered_df["차종"].unique())
    if "total_reviews" in filtered_df.columns:
        num_reviews = filtered_df["total_reviews"].sum()
    else:
        num_reviews = len(filtered_df)

    col1, col2 = st.columns(2)
    col1.metric("조건에 맞는 차종 수", num_unique_models)
    col2.metric("총 리뷰 수", num_reviews)

     # 전체 차량 목록 표시
    st.subheader("📋 전체 필터링된 차량 목록")

    if st.checkbox("전체 차량 목록 확인하기"):
        if filtered_df is not None and not filtered_df.empty:
            st.markdown(f"🔎 총 차량 수: {len(filtered_df['차종'].unique())}개")

            try:
                unique_cars = filtered_df.drop_duplicates(subset=["차종"])
                car_list_df = unique_cars[["차종", "제조사", "가격 (만원)", "연비(km/L)", "최고출력 (마력)", "배기량 (cc)"]].copy()
                car_list_df.columns = ["차종", "제조사", "가격(만원)", "연비(km/L)", "최고출력(마력)", "배기량(cc)"]
                st.dataframe(car_list_df)
            except KeyError as e:
                st.error(f"⚠️ 열 이름 오류: {e}. 실제 열 이름: {filtered_df.columns.tolist()}")
        else:
            st.info("⚠️ 조건에 맞는 차량이 없습니다.")


    # FAISS 인덱스 및 검색 구현
    if embedded_df is not None and "embedding" in filtered_df.columns:
        st.subheader("🔍 키워드 기반 차량 추천")
        search_query = st.text_input("추천받고 싶은 차량에 대한 리뷰나 키워드를 입력하세요:", "디자인이 예쁜차, 주행감이 좋은차")

        if st.button("차량 찾기"):
            # 리뷰가 있는 차량만 필터링
            review_filtered_df = filtered_df[filtered_df["리뷰"].notna() & (filtered_df["리뷰"].str.strip() != "")]

            if len(review_filtered_df) == 0:
                st.warning("⚠️ 조건에 맞는 유효한 리뷰가 없습니다. 필터링 조건을 조정해보세요.")
            else:
                # FAISS 인덱스 구성
                with st.spinner("🔄 검색 중..."):
                    # 임베딩 벡터 추출
                    car_review_vectors = np.vstack(review_filtered_df["embedding"].values).astype("float32")

                    # FAISS 인덱스 구성
                    faiss.normalize_L2(car_review_vectors)
                    d = car_review_vectors.shape[1]  # 임베딩 차원
                    index = faiss.IndexFlatIP(d)
                    index.add(car_review_vectors)

                    # 쿼리 벡터 생성
                    query_vector = model.encode(search_query, normalize_embeddings=True)
                    query_vector = np.array([query_vector], dtype="float32")
                    faiss.normalize_L2(query_vector)

                    # FAISS 검색
                    k = min(100, len(car_review_vectors))
                    D, I = index.search(query_vector, k)

                    # 유사도 점수 계산
                    similarities = D[0]
                    min_similarity = np.min(similarities) if len(similarities) > 0 else 0.0
                    max_similarity = np.max(similarities) if len(similarities) > 0 else 1.0

                    if max_similarity - min_similarity < 1e-5:
                        similarity_scores = np.ones_like(similarities)
                    else:
                        similarity_scores = (similarities - min_similarity) / (max_similarity - min_similarity)

                    # 차종 빈도 분석
                    top_indices = I[0]
                    top_car_models = [review_filtered_df.iloc[idx]["차종"] for idx in top_indices]

                    model_counter = Counter(top_car_models)

                    if not model_counter:
                        st.warning("⚠️ 검색 결과가 없습니다. 다른 키워드로 시도해보세요.")
                    else:
                        most_common_model, most_common_count = model_counter.most_common(1)[0]

                        # 빈도 기반 가중치 계산 함수
                        def get_model_frequency_score(model_name):
                            freq = model_counter[model_name]
                            return freq / most_common_count

                        # 차종별 감성 점수 저장
                        car_sentiment_scores = {}

                        # 필터링된 데이터프레임에서 차종별 감성 점수 추출
                        for idx, car_info in filtered_df.iterrows():
                            model_name = car_info["차종"]

                            if model_name in car_sentiment_scores:
                                continue

                            if "final_sentiment_score_new" in car_info:
                                sentiment_score_raw = car_info["final_sentiment_score_new"]
                                normalized_score = (sentiment_score_raw - 2) / (22 - 2) * 5
                                normalized_score = max(0, min(normalized_score, 5))
                                car_sentiment_scores[model_name] = normalized_score
                            else:
                                car_sentiment_scores[model_name] = 2.5

                        # 검색 결과 처리
                        recommendation_list = []
                        processed_cars = set()

                        for idx, sim_score in zip(top_indices, similarity_scores):
                            car_info = review_filtered_df.iloc[idx]
                            model_name = car_info["차종"]

                            if model_name in processed_cars:
                                continue

                            processed_cars.add(model_name)

                            sentiment_score = car_sentiment_scores.get(model_name, 2.5)
                            frequency_score = get_model_frequency_score(model_name) * 5
                            similarity_score_5 = sim_score * 5

                            final_score = (similarity_score_5 * 0.5 + sentiment_score * 0.3 + frequency_score * 0.2)

                            recommendation_list.append({
                                "차종": model_name,
                                "제조사": car_info["제조사"],
                                "가격": car_info["가격 (만원)"],
                                "연비": car_info["연비(km/L)"],
                                "배기량": car_info["배기량 (cc)"],
                                "최고출력": car_info["최고출력 (마력)"],
                                "감성점수": sentiment_score,
                                "유사도": similarity_score_5,
                                "차종빈도": frequency_score,
                                "종합점수": final_score,
                                "리뷰": car_info["리뷰"]
                            })

                            if len(processed_cars) >= 5:
                                break

                        # 결과가 5개보다 적을 경우 처리
                        if len(recommendation_list) < 5 and len(filtered_df["차종"].unique()) >= 5:
                            remaining_models = set(filtered_df["차종"].unique()) - processed_cars

                            for model in remaining_models:
                                model_rows = filtered_df[filtered_df["차종"] == model]
                                if model_rows.empty:
                                    continue

                                car_info = model_rows.iloc[0]
                                sentiment_score = car_sentiment_scores.get(model, 2.5)

                                recommendation_list.append({
                                    "차종": model,
                                    "제조사": car_info["제조사"],
                                    "가격": car_info["가격 (만원)"],
                                    "연비": car_info["연비(km/L)"],
                                    "배기량": car_info["배기량 (cc)"],
                                    "최고출력": car_info["최고출력 (마력)"],
                                    "감성점수": sentiment_score,
                                    "유사도": 1.0,
                                    "차종빈도": 1.0,
                                    "종합점수": 1.0 + (sentiment_score * 0.3),
                                    "리뷰": car_info["리뷰"] if isinstance(car_info["리뷰"], str) else ""
                                })

                                processed_cars.add(model)

                                if len(recommendation_list) >= 5:
                                    break

                        # 추천 결과 표시
                        if recommendation_list:
                            recommendation_list = sorted(recommendation_list, key=lambda x: x["종합점수"], reverse=True)

                            st.subheader("🏆 맞춤형 추천 차량 TOP 5")

                            # 추천 차량 정보 탭
                            info_tab, viz_tab = st.tabs(["🚗 추천 차량 정보", "📊 시각화 비교"])

                            with info_tab:
                                for i, car in enumerate(recommendation_list):
                                    st.markdown(f"### {i+1}. {car['차종']} ({car['제조사']})")
                                    col1, col2 = st.columns([2, 3])
                                    with col1:
                                        st.markdown(f"**💰 가격:** {car['가격']:,.0f}만원")
                                        st.markdown(f"**⛽ 연비:** {car['연비']:.1f} km/L")
                                        st.markdown(f"**🔧 배기량:** {car['배기량']}cc")
                                        st.markdown(f"**🏎️ 최고출력:** {car['최고출력']}마력")
                                        keywords = car_keywords.get(car['차종'], [])
                                        if keywords:
                                            st.markdown(f"**✨ 핵심 키워드:** {', '.join(keywords)}")
                                    with col2:
                                        score_df = pd.DataFrame({
                                            "항목": ["감성 점수", "유사도 점수", "차량 빈도", "종합 평가"],
                                            "점수": [car['감성점수'], car['유사도'], car['차종빈도'], car['종합점수']]
                                        })
                                        fig = px.bar(score_df, x="점수", y="항목", orientation='h', range_x=[0, 5],
                                                    color="점수", color_continuous_scale="RdYlGn")
                                        st.plotly_chart(fig, use_container_width=True)
                                        stars = "★" * int(car["종합점수"]) + "☆" * (5 - int(car["종합점수"]))
                                        st.markdown(f"### 종합 평점: {stars} ({car['종합점수']:.1f}점)")

                                    st.markdown("#### 💬 내 검색어와 유사한 사용자 리뷰")
                                    if isinstance(car['리뷰'], str) and car['리뷰'].strip():
                                        review_sample = car['리뷰'][:500] + "..." if len(car['리뷰']) > 500 else car['리뷰']
                                        st.text_area("리뷰", review_sample, height=150, key=f"review_{i}")
                                    else:
                                        st.info("이 차량에 대한 리뷰가 없습니다.")

                            with viz_tab:
                                st.subheader("🕸️ 차량별 성능 레이더 비교")
                                radar_data = recommendation_list[:3]
                                labels = ['종합점수', '연비', '배기량', '최고출력', '가격']
                                max_values = {label: max(car[label] for car in radar_data) for label in labels}

                                radar_fig = go.Figure()
                                for car in radar_data:
                                    values = [
                                        car['종합점수'] / max_values['종합점수'],
                                        car['연비'] / max_values['연비'],
                                        car['배기량'] / max_values['배기량'],
                                        car['최고출력'] / max_values['최고출력'],
                                        car['가격'] / max_values['가격']
                                    ]
                                    values += values[:1]
                                    radar_fig.add_trace(go.Scatterpolar(
                                        r=values,
                                        theta=labels + [labels[0]],
                                        fill='toself',
                                        name=f"{car['차종']} ({car['제조사']})"
                                    ))

                                radar_fig.update_layout(
                                    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
                                    showlegend=True
                                )
                                st.plotly_chart(radar_fig, use_container_width=True)

                                st.subheader("📊 추천 차량 비교표")
                                compare_df = pd.DataFrame([{
                                    "차종": car["차종"],
                                    "제조사": car["제조사"],
                                    "가격(만원)": car["가격"],
                                    "연비(km/L)": car["연비"],
                                    "배기량(cc)": car["배기량"],
                                    "최고출력(마력)": car["최고출력"],
                                    "종합점수": car["종합점수"]
                                } for car in recommendation_list])
                                st.dataframe(compare_df)


                                st.subheader("📝 추천 차량 키워드 Top 10")

                                keyword_counter = Counter()
                                for car in recommendation_list:
                                    model = car['차종']
                                    keywords = car_keywords.get(model, [])
                                    keyword_counter.update(keywords)

                                if keyword_counter:
                                    top_keywords = keyword_counter.most_common(10)
                                    words, freqs = zip(*top_keywords)

                                    fig = px.bar(x=words, y=freqs, title="추천 차량 키워드 빈도 Top 10")
                                    fig.update_layout(xaxis_title="키워드", yaxis_title="빈도")
                                    st.plotly_chart(fig, use_container_width=True)
                                else:
                                    st.info("추천된 차량들의 키워드 정보가 없습니다.")



                                st.subheader("☁️ 유사 리뷰 기반 워드클라우드")

                                # 1. FAISS 검색 결과 인덱스로 리뷰 텍스트 추출
                                similar_reviews = review_filtered_df.iloc[I[0]]["리뷰"]

                                # 2. 형태소 분석 + 불용어 제거
                                okt = Okt()
                                stopwords = set(["있다", "하다", "없다", "같다", "되다", "수", "것", "더", "좀", "이", "에", "를", "가", "은", "는", "의", "고"])

                                words = []
                                for text in similar_reviews:
                                    nouns = okt.nouns(text)
                                    filtered = [word for word in nouns if word not in stopwords and len(word) >= 2]
                                    words.extend(filtered)

                                # 3. 단어 빈도수 계산
                                raw_freq = Counter(words)

                                # 4. 최소 빈도 필터링 (예: 20 이상 등장한 단어만)
                                filtered_freq = {word: freq for word, freq in raw_freq.items() if freq >= 5}

                                # 5. 상위 k개 단어만 추출 (예: 최대 25개)
                                top_k = 50
                                word_freq = dict(Counter(filtered_freq).most_common(top_k))

                                # 6. 워드클라우드 생성 및 시각화
                                if word_freq:
                                    wordcloud = WordCloud(
                                        font_path=font_path,
                                        max_font_size=50,
                                        width=400,
                                        height=200,
                                        background_color='white'
                                    ).generate_from_frequencies(word_freq)

                                    fig, ax = plt.subplots(figsize=(10, 5))  # Streamlit에 작게 표시
                                    ax.imshow(wordcloud, interpolation='bilinear')
                                    ax.axis("off")
                                    plt.tight_layout(pad=0)
                                    st.pyplot(fig)
                                else:
                                    st.info("유사 리뷰에서 추출된 키워드가 없습니다.")



st.markdown("---")
st.subheader("📄 전체 통합 데이터 보기")
if st.checkbox("전체 데이터 확인"):
    st.dataframe(df)

Writing app.py


In [7]:
# 5. Streamlit 실행 및 ngrok 터널링 설정
import subprocess
import time
from pyngrok import ngrok

# 백그라운드에서 Streamlit 실행
proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py', '--server.headless=true', '--server.port=8501'],
    stdout=subprocess.PIPE
)

# ngrok 터널 생성
public_url = ngrok.connect(8501)
print(f"Streamlit 대시보드가 다음 URL에서 실행 중입니다: {public_url}")

# 로그 보기
try:
    for line in proc.stdout:
        print(line.decode("utf-8").strip())
except KeyboardInterrupt:
    proc.terminate()
    print("⛔ Streamlit 종료됨")


Streamlit 대시보드가 다음 URL에서 실행 중입니다: NgrokTunnel: "https://9f1d-34-168-200-159.ngrok-free.app" -> "http://localhost:8501"



You can now view your Streamlit app in your browser.

Local URL: http://localhost:8501
Network URL: http://172.28.0.12:8501
External URL: http://34.168.200.159:8501

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
fonts-nanum
0 upgraded, 1 newly installed, 0 to remove and 47 not upgraded.
Need to get 10.3 MB of archives.
After this operation, 34.1 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-nanum all 20200506-1 [10.3 MB]
Fetched 10.3 MB in 1s (12.8 MB/s)
Selecting previously unselected package fonts-nanum.
(Reading database ... 126213 files and directories currently installed.)
Preparing to unpack .../fonts-nanum_20200506-1_all.deb ...
Unpacking fonts-nanum (20200506-1) ...
Setting up fonts-nanum (20200506-1) ...
Processing 

In [12]:
from pyngrok import ngrok

# 현재 터널 목록 확인
tunnels = ngrok.get_tunnels()
print(f"현재 터널: {tunnels}")

현재 터널: []


In [13]:
# 또는 모든 터널 닫기
for tunnel in tunnels:
    print(f"터널 닫는 중: {tunnel.public_url}")
    ngrok.disconnect(tunnel.public_url)